In [ ]:
# Install required packages
!pip install python-jose cryptography langchain-core langchain-community langchain-openai langchain-text-splitters langchain-experimental faiss-cpu python-dotenv pyyaml numpy pandas pypdf PyMuPDF rank-bm25

# Final Evaluation – A0 through A8

Discovers all experiment JSONL results and produces summary tables for Chapter 4.

In [ ]:
import sys, json
import numpy as np, pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from report_common.io import load_jsonl, save_csv_summary, discover_experiments, build_comparison_table, build_category_breakdown, build_pairwise_deltas

In [ ]:
results_dir = PROJECT_ROOT / 'report_results'
experiments = discover_experiments(results_dir)

print(f'Discovered {len(experiments)} experiment files:')
for name, records in experiments.items():
    print(f'  {name}: {len(records)} records')

# Warning if fewer than 9 experiments (A0-A8)
expected = {f'A{i}' for i in range(9)}
found_prefixes = set()
for name in experiments:
    for prefix in expected:
        if name.startswith(prefix):
            found_prefixes.add(prefix)
missing = expected - found_prefixes
if missing:
    print(f'\n\u26a0\ufe0f  WARNING: Missing experiments: {sorted(missing)}')
    print(f'   Only {len(found_prefixes)}/9 experiments found. Run missing notebooks first.')

In [ ]:
df = build_comparison_table(experiments)
print(df.to_string(index=False))

## Results by Question Category

In [ ]:
with open(PROJECT_ROOT / 'report_data/evaluation/questions.json', 'r') as f:
    questions = json.load(f)

for exp_name, records in experiments.items():
    print(f'\n{"="*60}')
    print(f'{exp_name} - By Category')
    print(f'{"="*60}')
    cat_df = build_category_breakdown(records, questions)
    if not cat_df.empty:
        print(cat_df.to_string(index=False))

## Pairwise Delta vs Baseline (A0)

In [ ]:
delta_df = build_pairwise_deltas(experiments)
if not delta_df.empty:
    print(delta_df.to_string(index=False))
else:
    print('Need A0 baseline results to compute deltas.')

In [ ]:
output_dir = results_dir / 'summary'
output_dir.mkdir(parents=True, exist_ok=True)
save_csv_summary(df.to_dict('records'), output_dir / 'all_experiments.csv')
print('Summary saved to report_results/summary/all_experiments.csv')